latest parser version with ALL available options

In [ ]:
# Imports
import os
import sys
from IPython.display import display
from pathlib import Path

# Find the nearest directory that contains 'py_scripts' and add it to sys.path
cwd = Path(os.getcwd()).resolve()
for base in [cwd, *cwd.parents]:
    if (base / "py_scripts").is_dir():
        sys.path.insert(0, str(base))
        break
else:
    raise ImportError(f"Could not find a 'py_scripts' directory above {cwd}")

from py_scripts.parser_registry import parse_files


def expand_file_sources(sources: list) -> list:
    """
    Expand FILE_SOURCES list by reading any .txt files and extracting URLs/paths.
    
    - If a source ends with '.txt', read the file and extract all non-empty, 
      non-comment lines (lines not starting with '#').
    - Supports both relative paths (resolved from notebook location) and absolute paths.
    - Other sources (URLs, direct file paths) are passed through unchanged.
    
    Returns a flat list of all expanded sources.
    """
    expanded = []
    notebook_dir = Path(os.getcwd()).resolve()
    
    for source in sources:
        source = source.strip()
        if not source:
            continue
            
        # Check if this is a .txt file (not a URL)
        if source.lower().endswith('.txt') and not source.startswith(('http://', 'https://')):
            # Resolve the path (handle both relative and absolute)
            txt_path = Path(source)
            if not txt_path.is_absolute():
                txt_path = notebook_dir / txt_path
            txt_path = txt_path.resolve()
            
            if txt_path.exists():
                count_before = len(expanded)
                print(f"📄 Expanding sources from: {txt_path}")
                with open(txt_path, 'r', encoding='utf-8') as f:
                    for line in f:
                        line = line.strip()
                        # Skip empty lines and comments
                        if line and not line.startswith('#'):
                            expanded.append(line)
                print(f"   → Found {len(expanded) - count_before} sources")
            else:
                print(f"⚠️ Warning: txt file not found: {txt_path}")
        else:
            # Regular URL or file path - pass through
            expanded.append(source)
    
    return expanded


# Backend selection ('music21' | 'partitura')
PARSING_BACKEND = 'partitura'

# Configuration
FILTER_ZERO_DURATION = True # filter out notes with duration 0 (grace notes)
ADJUST_FRACTIONAL_DURATION = True # rounds Duration, Local Onset, and Global Onset to 3 decimals
PARSE_ENHARMONIC = True     # add 'Pitch Enharmonic' column grounded in source notation
PLOTTING_BACKEND = 'bokeh'  # 'plt' | 'bokeh' | 'none'

SHOW_MEASURE_LINES = True   # draw vertical measure lines at measure starts
MEASURE_LINE_COLOR = 'black'  # color for the measure lines

HOVER_ENABLED = True        # enable hover tool (Bokeh only)
HOVER_FIELDS = ['measure', 'local_onset', 'global_onset', 'duration', 'pitch', 'pitch_enharmonic', 'midi', 'voice', 'xml_id'] # hover options as list

DISPLAY_PREVIEW = True      # show a preview of the parsed data as df
PREVIEW_ROWS = 20          # number of rows to show in the preview

CLEANUP_REMOTE = True       # if source is a URL, delete temp file after parsing
RETURN_PLOTS = False        # include plot objects in results under 'plot'
STRIP_TIES = True           # merge tied notes into single notes (music21 backend only)
ALIGN_ACCIDENT_SCHEMA = True # align accidentals on both parsing backends to the same schema (Pitch Enharmonic is parsed only in b/# format)

COLORIZE_VOICES = True      # color-code notes by voice/part in the piano roll
PALETTE = 'Category20'      # palette name ('Category10'/'tab10', 'Category20'/'tab20', 'Set3', or a list)
INCLUDE_XML_IDS = True      # if MEI, include xml:id per note as 'xml_id' column

# List of file sources (URLs or local paths)

# USAGE EXAMPLES:

# Use the txt file with relative path (relative to notebook location)
# FILE_SOURCES = [
#     '../../test_corpus/test_corpus_links.txt'
# ]

# # Or with absolute path
# FILE_SOURCES = [
#     'C:/Users/egorp/Nextcloud/code/public_repos/camat_v2/test_corpus/test_corpus_links.txt'
# ]

# # Mix txt files with direct URLs
# FILE_SOURCES = [
#     '../../test_corpus/test_corpus_links.txt',
#     'https://raw.githubusercontent.com/music-encoding/sample-encodings/main/MEI_5.0/Music/Complete_examples/Bach-JS_Ein_feste_Burg.mei'
# ]


FILE_SOURCES = [
    '../../test_corpus/test_corpus_links.txt'
    # 'https://raw.githubusercontent.com/music-encoding/sample-encodings/main/MEI_5.0/Music/Complete_examples/Bach-JS_Ein_feste_Burg.mei',
    # 'https://raw.githubusercontent.com/music-encoding/sample-encodings/main/MEI_3.0/Music/Complete_examples/Mozart_Fuge_G_minor.mei',
    # 'C:/Users/egorp/OneDrive/Desktop/weimar_ftp_backup/dokuwiki/database/05/CaSe_11_UNSP_UNSP_Danksagenw_005_00011.xml'  # If you prefer backslashes, make it a raw string: r'C\Users\egorp\...'
]

# Expand FILE_SOURCES (resolves .txt files into their contained URLs/paths)
EFFECTIVE_FILE_SOURCES = expand_file_sources(FILE_SOURCES)

# Plot sizing
PLOT_SIZE_X = 900  # width in pixels
PLOT_SIZE_Y = 600   # height in pixels

# Zoom tool dimensions: 'both' | 'width' (x-only) | 'height' (y-only)
# Bokeh only!
ZOOM_DRAG_DIM = 'both'
ZOOM_WHEEL_DIM = 'width'

# Parse and visualize
results, dfs_by_name, df_processed = parse_files(
    EFFECTIVE_FILE_SOURCES,
    parsing_backend=PARSING_BACKEND,
    filter_zero_duration=FILTER_ZERO_DURATION,
    adjust_fractional_duration=ADJUST_FRACTIONAL_DURATION,
    parse_enharmonic=PARSE_ENHARMONIC,
    backend=PLOTTING_BACKEND,
    show_measure_lines=SHOW_MEASURE_LINES,
    measure_line_color=MEASURE_LINE_COLOR,
    show_hover=HOVER_ENABLED,
    hover_fields=HOVER_FIELDS,
    display_preview=DISPLAY_PREVIEW,
    preview_rows=PREVIEW_ROWS,
    cleanup_remote=CLEANUP_REMOTE,
    return_plots=RETURN_PLOTS,
    plot_width=PLOT_SIZE_X,
    plot_height=PLOT_SIZE_Y,
    zoom_drag_dim=ZOOM_DRAG_DIM,
    zoom_wheel_dim=ZOOM_WHEEL_DIM,
    strip_ties=STRIP_TIES,
    align_accident_schema=ALIGN_ACCIDENT_SCHEMA,
    colorize_voices=COLORIZE_VOICES,
    palette=PALETTE,
    include_xml_ids=INCLUDE_XML_IDS
)

# Show quick summary
print('Parsed DataFrames:')
for item in results:
    name = item['name']
    df = item['df']
    pos_dur = df.loc[df['Duration'] > 0, 'Duration'] if 'Duration' in df.columns else None
    min_dur_str = str(float(pos_dur.min())) if pos_dur is not None and len(pos_dur) > 0 else 'n/a'
    voice_count = df['Voice'].nunique() if 'Voice' in df.columns else 0
    voice_names = sorted(df['Voice'].unique().tolist()) if 'Voice' in df.columns else []
    print(f"- {name}: rows={len(df)}, unique_pitches={df['MIDI'].nunique()}, min_duration={min_dur_str}, voices={voice_count} ({voice_names})")


📄 Expanding sources from: C:\Users\egorp\Nextcloud\code\public_repos\camat_v2\test_corpus\test_corpus_links.txt
   → Found 40 sources


Parsing files:   0%|          | 0/40 [00:00<?, ?file/s]

Processing (partitura): sonata29-1.krn -> 00_sonata29_1
An error occurred while processing https://raw.githubusercontent.com/craigsapp/beethoven-piano-sonatas/master/kern/sonata29-1.krn: list index out of range
Processing (partitura): sonata29-2.krn -> 01_sonata29_2


c:\Users\egorp\miniconda3\envs\py312latest\Lib\site-packages\partitura\io\importkern.py:552: UserWarning: Slurs openings and closings do not match. Skipping parsing slurs for this part P0.
  warnings.warn(
c:\Users\egorp\miniconda3\envs\py312latest\Lib\site-packages\partitura\io\importkern.py:355: UserWarning: Part P0 already exists. Adding to previous Part.
  warnings.warn(


,Measure,Local Onset,Global Onset,Duration,Pitch,Pitch Enharmonic,MIDI,Voice,xml_id
0,0,2.00,-1.00,0.75,A#3,Bb3,58,P0 - Voice 1,<NA>
1,0,2.00,-1.00,0.75,D4,D4,62,P0 - Voice 1,<NA>
2,0,2.00,-1.00,0.75,D5,D5,74,P0 - Voice 3,<NA>
3,0,2.75,-0.25,0.25,F3,F3,53,P0 - Voice 1,<NA>
4,0,2.75,-0.25,0.25,C4,C4,60,P0 - Voice 1,<NA>
5,0,2.75,-0.25,0.25,A4,A4,69,P0 - Voice 3,<NA>
6,0,2.75,-0.25,0.25,F5,F5,77,P0 - Voice 3,<NA>
7,0,0.00,0.00,1.00,A3,A3,57,P0 - Voice 1,<NA>
8,0,0.00,0.00,1.00,C4,C4,60,P0 - Voice 1,<NA>
9,0,0.00,0.00,1.00,C5,C5,72,P0 - Voice 3,<NA>


...

,Measure,Local Onset,Global Onset,Duration,Pitch,Pitch Enharmonic,MIDI,Voice,xml_id
0,1,0.000,0.000,0.164,C4,C4,60,Piano - Voice 2,<NA>
1,1,0.000,0.000,0.500,C6,C6,84,Piano - Voice 1,<NA>
2,1,0.164,0.164,0.168,D#4,Eb4,63,Piano - Voice 2,<NA>
3,1,0.332,0.332,0.168,G4,G4,67,Piano - Voice 2,<NA>
4,1,0.500,0.500,0.164,D4,D4,62,Piano - Voice 2,<NA>
5,1,0.500,0.500,0.500,B5,B5,83,Piano - Voice 1,<NA>
6,1,0.664,0.664,0.168,F4,F4,65,Piano - Voice 2,<NA>
7,1,0.832,0.832,0.168,G4,G4,67,Piano - Voice 2,<NA>
8,1,1.000,1.000,0.500,C6,C6,84,Piano - Voice 1,<NA>
9,1,1.000,1.000,0.164,D#4,Eb4,63,Piano - Voice 2,<NA>


Rows: 739, unique pitches: 50
Processing (partitura): Schubert_D911-07.xml -> 39_schubert_d911_07


c:\Users\egorp\miniconda3\envs\py312latest\Lib\site-packages\partitura\directions.py:514: UserWarning: error parsing "Langsam." (UnexpectedCharacters)
  warnings.warn('error parsing "{}" ({})'.format(string, type(e).__name__))
c:\Users\egorp\miniconda3\envs\py312latest\Lib\site-packages\partitura\io\importmusicxml.py:1065: UserWarning: ignoring direction type: metronome {'default-y': '30', 'color': '#000000', 'font-family': 'Plantin MT Std', 'font-style': 'italic', 'font-size': '9.3012', 'font-weight': 'bold'}
  warnings.warn("ignoring direction type: {} {}".format(dt.tag, dt.attrib))
c:\Users\egorp\miniconda3\envs\py312latest\Lib\site-packages\partitura\directions.py:514: UserWarning: error parsing "(sehr leise)" (UnexpectedCharacters)
  warnings.warn('error parsing "{}" ({})'.format(string, type(e).__name__))
c:\Users\egorp\miniconda3\envs\py312latest\Lib\site-packages\partitura\io\importmusicxml.py:1614: UserWarning: Tuplet start and end notes do not belong to the same voice (2 != 1

,Measure,Local Onset,Global Onset,Duration,Pitch,Pitch Enharmonic,MIDI,Voice,xml_id
0,1,0.0,0.0,0.5,E3,E3,52,Piano - Voice 2,<NA>
1,1,0.5,0.5,0.5,B3,B3,59,Piano - Voice 1,<NA>
2,1,0.5,0.5,0.5,E4,E4,64,Piano - Voice 1,<NA>
3,1,0.5,0.5,0.5,G4,G4,67,Piano - Voice 1,<NA>
4,1,1.0,1.0,0.5,D3,D3,50,Piano - Voice 2,<NA>
5,1,1.5,1.5,0.5,B3,B3,59,Piano - Voice 1,<NA>
6,1,1.5,1.5,0.5,E4,E4,64,Piano - Voice 1,<NA>
7,1,1.5,1.5,0.5,G4,G4,67,Piano - Voice 1,<NA>
8,1,2.0,2.0,0.5,C3,C3,48,Piano - Voice 2,<NA>
9,1,2.5,2.5,0.5,C4,C4,60,Piano - Voice 1,<NA>


Rows: 1330, unique pitches: 42
Parsed DataFrames:
- 01_sonata29_2: rows=2001, unique_pitches=70, min_duration=0.25, voices=4 (['P0 - Voice 1', 'P0 - Voice 2', 'P0 - Voice 3', 'P0 - Voice 4'])
- 02_sonata29_3: rows=5265, unique_pitches=70, min_duration=0.083, voices=4 (['P0 - Voice 1', 'P0 - Voice 2', 'P0 - Voice 3', 'P0 - Voice 4'])
- 03_sonata29_4: rows=6525, unique_pitches=71, min_duration=0.062, voices=4 (['P0 - Voice 1', 'P0 - Voice 2', 'P0 - Voice 3', 'P0 - Voice 4'])
- 04_sonata32_1: rows=4063, unique_pitches=73, min_duration=0.083, voices=4 (['P0 - Voice 1', 'P0 - Voice 2', 'P0 - Voice 3', 'P0 - Voice 4'])
- 05_sonata32_2: rows=6012, unique_pitches=70, min_duration=0.042, voices=4 (['P0 - Voice 1', 'P0 - Voice 2', 'P0 - Voice 3', 'P0 - Voice 4'])
- 06_sonata31_1: rows=2712, unique_pitches=69, min_duration=0.125, voices=4 (['P0 - Voice 1', 'P0 - Voice 2', 'P0 - Voice 3', 'P0 - Voice 4'])
- 07_sonata31_2: rows=966, unique_pitches=62, min_duration=0.5, voices=4 (['P0 - Voice 1', 'P